# MISCFilter Deblurring

## 1. Environment Setup

In [ ]:
import os

REPO_URL = "https://github.com/ChengxuLiu/MISCFilter.git"

if not os.path.exists('models') and not os.path.exists('utils'):
    !git clone {REPO_URL}
    repo_name = REPO_URL.split('/')[-1].replace('.git', '')
    if os.path.exists(repo_name):
        %cd {repo_name}
else:
    if os.path.basename(os.getcwd()) != 'MISCFilter' and os.path.exists('MISCFilter'):
        %cd MISCFilter

print(f"Working Directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
import torch

!pip install yacs joblib natsort kornia ptflops gdown opencv-python scikit-image matplotlib

if os.path.exists('pytorch-gradual-warmup-lr'):
    !pip install ./pytorch-gradual-warmup-lr
else:
    !pip install git+https://github.com/ildoonet/pytorch-gradual-warmup-lr.git

cuda_version = torch.version.cuda
print(f"CUDA Version: {cuda_version}")
if cuda_version:
    major = cuda_version.split('.')[0]
    if major == '12':
        !pip install cupy-cuda12x
    elif major == '11':
        !pip install cupy-cuda11x
    else:
        !pip install cupy
else:
    print("Error: GPU acceleration not active. Enable GPU in settings.")

## 3. Setup Pretrained Weights

In [ ]:
WEIGHT_FILE = "RealBlur_J.pth"
WEIGHTS_PATH = ""

if os.path.exists(WEIGHT_FILE):
    WEIGHTS_PATH = f"./{WEIGHT_FILE}"
    print(f"Using local root weights: {WEIGHTS_PATH}")
elif os.path.exists(os.path.join("checkpoints", WEIGHT_FILE)):
    WEIGHTS_PATH = f"./checkpoints/{WEIGHT_FILE}"
    print(f"Using local checkpoint weights: {WEIGHTS_PATH}")
elif os.path.exists("checkpoints") and len(os.listdir("checkpoints")) > 0:
    available_weights = [f for f in os.listdir("checkpoints") if f.endswith('.pth')]
    if available_weights:
        WEIGHTS_PATH = f"./checkpoints/{available_weights[0]}"
        print(f"Using local fallback weights: {WEIGHTS_PATH}")

if not WEIGHTS_PATH:
    print("Weights not found locally. Downloading from Google Drive...")
    os.makedirs('checkpoints', exist_ok=True)
    !pip install --upgrade gdown
    !gdown --folder 1M-Sc_u97vTQskfO6VMzPghb8LnerP8Id -O checkpoints/
    
    if os.path.exists(os.path.join("checkpoints", WEIGHT_FILE)):
        WEIGHTS_PATH = f"./checkpoints/{WEIGHT_FILE}"
    else:
        WEIGHTS_PATH = "./checkpoints/GoPro.pth"

print(f"Final weights path: {WEIGHTS_PATH}")

## 4. Run Deblurring

In [ ]:
os.makedirs('test_images', exist_ok=True)

!python inference_custom.py \
    --input_dir ./test_images \
    --output_dir ./results \
    --weights {WEIGHTS_PATH} \
    --win_size 256 \
    --chunk_size 8 \
    --inference_mode normal \
    --limit 3

## 5. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

blurred_dir = './test_images'
restored_dir = './results'

restored_files = [f for f in sorted(os.listdir(restored_dir)) if f.endswith('_deblurred.png')]
if restored_files:
    restored_name = restored_files[0]
    original_name = restored_name.replace('_deblurred.png', '')
    original_file = None
    for ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp']:
        potential_path = os.path.join(blurred_dir, original_name + ext)
        if os.path.exists(potential_path):
            original_file = original_name + ext
            break
            
    if original_file:
        img_blur = Image.open(os.path.join(blurred_dir, original_file))
        img_restore = Image.open(os.path.join(restored_dir, restored_name))
        
        fig, axes = plt.subplots(1, 2, figsize=(18, 9))
        axes[0].imshow(img_blur)
        axes[0].set_title(f'Blurred ({original_file})')
        axes[0].axis('off')
        
        axes[1].imshow(img_restore)
        axes[1].set_title(f'Deblurred ({restored_name})')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("Matching original file not found.")
else:
    print("No results found in ./results.")

## 6. Zip & Download

In [ ]:
!zip -r deblurred_results.zip results/

try:
    from google.colab import files
    files.download('deblurred_results.zip')
except ImportError:
    print("Finished. Download 'deblurred_results.zip' using your browser sidebar.")